In [ ]:
#Importing necessary libraries for data handling, modeling, handling class imbalance, and evaluating.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV

#Loading the training and testing datasets.
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv")
y_test = pd.read_csv("data/y_test.csv")

#Converting y_train and y_test to 1D arrays for modeling.
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [2]:
#Getting the unique action classes from the training labels.
actions = np.sort(np.unique(y_train))

#Printing the action classes and the number of classes.
print(actions)
print(f'The number of unique classes is {len(actions)}.')

['appear' 'click' 'disappear' 'drag' 'hover' 'scrolldown' 'scrollup'
 'select' 'type' 'zoomin' 'zoomout']
The number of unique classes is 11.


In [3]:
#Defining the rare classes.
rare_classes = ["type", "hover", "scrolldown", "appear", "disappear", "scrollup"]

#Replacing rare classes with "other" in the training labels.
y_train = pd.Series(y_train).replace(rare_classes, "other").values

#Replacing rare classes with "other" in the testing labels.
y_test = pd.Series(y_test).replace(rare_classes, "other").values

#Getting the unique action classes from the training labels.
actions = np.sort(np.unique(y_train))

In [11]:
#Storing the best hyperparameters found from GridSearchCV for each class.
best_params = {"click":   {"C": 10.0, "l1_ratio": 0.7, "threshold": 0.540009},
"drag": {"C": 1.0, "l1_ratio": 0.7, "threshold": 0.411805},
"other": {"C": 10.0, "l1_ratio": 0.3, "threshold": 0.869458},
"select": {"C": 10.0, "l1_ratio": 0.3, "threshold": 0.969807},
"zoomin": {"C": 10.0, "l1_ratio": 0.3, "threshold": 0.907113},
"zoomout": {"C": 10.0, "l1_ratio": 0.5, "threshold": 0.812558}}

#Storing the trained models and evaluation metrics for each class.
ova_models = {}
ova_results = {}

#Looping through each action class to build the ova models.
for action in actions:
    
    #Creating binary training and testing labels where the current class is 1 and all others are 0.
    y_train_binary = (y_train == action).astype(int)
    y_test_binary = (y_test == action).astype(int)
    
    #Applying SMOTE to balance the training data.
    smote = SMOTE(random_state = 42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train_binary)
    
    #Scaling the features after SMOTE.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_resampled)
    X_test_scaled = scaler.transform(X_test)
    
    #Training an Elastic Net logistic regression model.
    model = LogisticRegression(penalty = "elasticnet", solver = "saga",
    C = best_params[action]["C"], l1_ratio = best_params[action]["l1_ratio"],
    max_iter = 2000, random_state = 42)
    model.fit(X_train_scaled, y_train_resampled)
    
    #Storing the trained model for the current class.
    ova_models[action] = model
    
    #Predicting probabilities on the test set.
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    #Applying the best threshold for the specific class.
    y_pred = (y_prob >= best_params[action]["threshold"]).astype(int)
    
    #Calculating evaluation metrics for the current class.
    accuracy = accuracy_score(y_test_binary, y_pred)
    precision = precision_score(y_test_binary, y_pred)
    recall = recall_score(y_test_binary, y_pred)
    f1 = f1_score(y_test_binary, y_pred)
    
    #Storing the evaluation metrics for the current class.
    ova_results[action] = {"Accuracy": accuracy, "Precision": precision,
    "Recall": recall, "F1": f1}

# Converting the stored results into a fresh DataFrame.
results_df = pd.DataFrame(ova_results).T

# Printing the fresh results DataFrame.
print(results_df)

c:\Users\dango\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\dango\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\dango\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\dango\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\dango\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the 

         Accuracy  Precision    Recall        F1
click    0.691193   0.518041  0.690722  0.592047
drag     0.705686   0.652830  0.812207  0.723849
other    0.949833   0.195122  0.400000  0.262295
select   0.966555   0.266667  0.173913  0.210526
zoomin   0.967670   0.815385  0.757143  0.785185
zoomout  0.971014   0.788732  0.835821  0.811594


c:\Users\dango\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
"""
#Performing a stratified split on the training data.
X_train_fold, X_val, y_train_fold, y_val = train_test_split(X_train, y_train, 
test_size = 0.2, random_state = 42, stratify = y_train)

#Storing the trained models, scalers, thresholds, and evaluation results for each class.
ova_models = {}
ova_scalers = {}
ova_thresholds = {}
ova_results = {}

#Looping through each action class to build the ova models.
for action in actions:
    
    #Creating binary labels where the current class is 1 and all other classes are 0.
    y_train_binary = (y_train_fold == action).astype(int)
    y_val_binary = (y_val == action).astype(int)
    y_test_binary = (y_test == action).astype(int)
    
    #Applying SMOTE to the training fold to balance the binary classes.
    smote = SMOTE(random_state = 42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_fold, y_train_binary)
    
    #Scaling the resampled training data and then transforming the validation and test sets.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_resampled)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    #Creating the Elastic Net logistic regression model.
    base_model = LogisticRegression(penalty = 'elasticnet', solver = 'saga',
    max_iter = 3000, random_state = 42)
    
    #Defining the smaller hyperparameter grid.
    param_grid = {'C': [0.1, 1, 10], 'l1_ratio': [0.3, 0.5, 0.7]}
    
    #Running grid search on the training fold to find the best hyperparameters.
    grid = GridSearchCV(estimator = base_model, param_grid = param_grid, scoring = 'f1',
    cv = 3, n_jobs = -1)
    
    #Fitting the grid search and storing the best model for the current class.
    grid.fit(X_train_scaled, y_train_resampled)
    best_model = grid.best_estimator_
    
    #Predicting validation probabilities to determine the best classification threshold.
    y_val_probs = best_model.predict_proba(X_val_scaled)[:, 1]
    
    #Computing precision, recall, and threshold values from the validation probabilities.
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_val_binary, y_val_probs)
    
    #Calculating the F1 score at each threshold.
    f1_scores = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (
        precision_vals[:-1] + recall_vals[:-1] + 1e-8)
    
    #Selecting the threshold that produces the highest validation F1 score.
    best_index = np.argmax(f1_scores)
    best_threshold = thresholds[best_index]
    
    #Storing the best model, scaler, and threshold for the current class.
    ova_models[action] = best_model
    ova_scalers[action] = scaler
    ova_thresholds[action] = best_threshold
    
    #Predicting probabilities on the test set and applying the chosen threshold.
    y_test_probs = best_model.predict_proba(X_test_scaled)[:, 1]
    y_test_pred = (y_test_probs >= best_threshold).astype(int)
    
    #Calculating the final test metrics for the current class.
    accuracy = accuracy_score(y_test_binary, y_test_pred)
    precision = precision_score(y_test_binary, y_test_pred)
    recall = recall_score(y_test_binary, y_test_pred)
    f1 = f1_score(y_test_binary, y_test_pred)
    
    #Storing the final metrics and best hyperparameters for the current class.
    ova_results[action] = {"Accuracy": accuracy, "Precision": precision,
    "Recall": recall,"F1": f1, "Best Threshold": best_threshold,
    "Best C": grid.best_params_["C"], "Best l1_ratio": grid.best_params_["l1_ratio"]}

#Converting the stored results into a DataFrame.
results_df = pd.DataFrame(ova_results).T

#Printing the results.
print(f'Final One-vs-All Results:')
print(results_df)


KeyboardInterrupt: 